# MASTER — Run the ECG benchmark experiments

This is the single Colab entry point for all Week 2 architecture work and the Week 3 matrices:

1. Validate every available dataset against the frozen manifest and signal contract.
2. Train **InceptionTime**, **ResNet1D**, and the vanilla **Transformer** from scratch on every ready source dataset.
3. Fine-tune **ECG-FM** on the same sources using the Week 1 adapter.
4. Record in-distribution macro-AUROC and per-class AUROC for all four architectures.
5. Run all four source-by-target evaluation matrices and the proposal's composite Score.
6. Retrain **ECG-FM** and **InceptionTime** for the two-class normal-vs-abnormal ablation.
7. Run both two-class matrices and calculate how much of the five-class gap disappears.

One Colab GPU cannot safely train multiple large models simultaneously. This notebook queues the experiments sequentially, saves every checkpoint/result to Drive, and resumes completed runs after a disconnect.

The old shared notebook is not used because it contains mock label mappings. This notebook imports the validated pipelines from the repository and writes every run to the same resumable Drive output layout.


In [ ]:
#@title 1. Run controls
MODE = "smoke"  #@param ["smoke", "full"]
RUN_DATA_PREPARATION = True  #@param {type:"boolean"}
RUN_SANITY_CHECKS = True  #@param {type:"boolean"}
RUN_ALL_5_CLASS_TRAINING = True  #@param {type:"boolean"}
RUN_ALL_5_CLASS_MATRICES = True  #@param {type:"boolean"}
RUN_COMPOSITE_SCORE = True  #@param {type:"boolean"}
RUN_2_CLASS_TRAINING = True  #@param {type:"boolean"}
RUN_2_CLASS_MATRIX = True  #@param {type:"boolean"}
RESUME_COMPLETED_RUNS = True  #@param {type:"boolean"}
STRICTLY_REQUIRE_ALL_FIVE_DATASETS = False  #@param {type:"boolean"}
STAGE_NPY_SIGNALS_TO_LOCAL_DISK = True  #@param {type:"boolean"}
PROJECT_ROOT_OVERRIDE = ""  #@param {type:"string"}
SHIFT_TABLE_OVERRIDE = ""  #@param {type:"string"}

REQUESTED_DATASETS = ["ptbxl", "cpsc2018", "georgia", "mimic_iv", "code_ii"]
FIVE_CLASS_ARCHITECTURES = ["inception_time", "resnet1d", "transformer", "ecg_fm"]
BINARY_ARCHITECTURES = ["ecg_fm", "inception_time"]
SEED = 42

if MODE not in {"smoke", "full"}:
    raise ValueError("MODE must be 'smoke' or 'full'")

print({
    "mode": MODE,
    "requested_datasets": REQUESTED_DATASETS,
    "five_class_architectures": FIVE_CLASS_ARCHITECTURES,
    "binary_architectures": BINARY_ARCHITECTURES,
    "resume": RESUME_COMPLETED_RUNS,
})


In [ ]:
#@title 2. Mount Drive and locate the reorganized project
from google.colab import drive
drive.mount("/content/drive")

import os
from pathlib import Path

PROJECT_FOLDER_NAME = "LSTS ECG Generalization Benchmark — START HERE"

def locate_project_root():
    if PROJECT_ROOT_OVERRIDE:
        path = Path(PROJECT_ROOT_OVERRIDE).expanduser()
        if not path.is_dir():
            raise FileNotFoundError(f"PROJECT_ROOT_OVERRIDE does not exist: {path}")
        return path

    direct_candidates = [
        Path("/content/drive/MyDrive") / PROJECT_FOLDER_NAME,
        Path("/content/drive/Shareddrives") / PROJECT_FOLDER_NAME,
    ]
    for candidate in direct_candidates:
        if candidate.is_dir():
            return candidate

    for search_root in (Path("/content/drive/Shareddrives"), Path("/content/drive/MyDrive")):
        if not search_root.exists():
            continue
        for base, directories, _ in os.walk(search_root):
            if PROJECT_FOLDER_NAME in directories:
                return Path(base) / PROJECT_FOLDER_NAME
    raise FileNotFoundError(
        f"Could not locate '{PROJECT_FOLDER_NAME}'. Set PROJECT_ROOT_OVERRIDE in cell 1."
    )

PROJECT_ROOT = locate_project_root()
DATASETS_ROOT = PROJECT_ROOT / "01_DATASETS"
CODE_ROOT = PROJECT_ROOT / "02_CODE_AND_NOTEBOOKS"
RESULTS_ROOT = PROJECT_ROOT / "03_RESULTS_AND_MODEL_OUTPUTS"

for required in (DATASETS_ROOT, CODE_ROOT, RESULTS_ROOT):
    if not required.is_dir():
        raise FileNotFoundError(f"Missing required project folder: {required}")

print({"project_root": str(PROJECT_ROOT)})


In [ ]:
#@title 3. Install the validated repository and ECG-FM dependency
import base64
import subprocess
import sys
from google.colab import userdata

REPOSITORY_URL = "https://github.com/tanushappapogu-max/ecg-generalization-benchmark.git"
WORKSPACE = Path("/content/ecg-generalization-benchmark")

try:
    github_token = userdata.get("GITHUB_TOKEN")
except Exception:
    github_token = None
if not github_token:
    raise RuntimeError(
        "Missing Colab secret GITHUB_TOKEN. Add it under the key icon (Secrets) "
        "and enable Notebook access. The token only needs Contents: Read-only."
    )

encoded_auth = base64.b64encode(f"x-access-token:{github_token}".encode()).decode()
git_env = {
    **os.environ,
    "GIT_TERMINAL_PROMPT": "0",
    "GIT_CONFIG_COUNT": "1",
    "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
    "GIT_CONFIG_VALUE_0": f"AUTHORIZATION: basic {encoded_auth}",
}

if WORKSPACE.exists() and not (WORKSPACE / ".git").is_dir():
    raise RuntimeError(
        f"{WORKSPACE} is a partial checkout from an earlier failed clone. "
        "Choose Runtime > Restart session, then Run all."
    )

if not WORKSPACE.exists():
    subprocess.check_call(
        ["git", "clone", "--depth", "1", "--branch", "main", REPOSITORY_URL, str(WORKSPACE)],
        env=git_env,
    )
else:
    subprocess.check_call(
        ["git", "-C", str(WORKSPACE), "pull", "--ff-only"],
        env=git_env,
    )

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "-e", str(WORKSPACE), "huggingface_hub", "wfdb", "scikit-learn", "pandas", "pytest",
])

FAIRSEQ_SIGNALS = Path("/content/fairseq-signals")
if not FAIRSEQ_SIGNALS.exists():
    subprocess.check_call([
        "git", "clone", "--depth", "1",
        "https://github.com/Jwoo5/fairseq-signals.git", str(FAIRSEQ_SIGNALS),
    ])
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(FAIRSEQ_SIGNALS)],
    env={**os.environ, "MAX_JOBS": "2"},
)

sys.path.insert(0, str(WORKSPACE))
print({"repository": str(WORKSPACE), "branch": "main"})


In [ ]:
#@title 3a. Verify this run uses the current GitHub main
def git_output(*arguments):
    return subprocess.check_output(
        ["git", "-C", str(WORKSPACE), *arguments],
        env=git_env,
        text=True,
    ).strip()

subprocess.check_call(
    ["git", "-C", str(WORKSPACE), "fetch", "--prune", "origin", "main"],
    env=git_env,
)
local_commit = git_output("rev-parse", "HEAD")
remote_commit = git_output("rev-parse", "origin/main")

if local_commit != remote_commit:
    raise RuntimeError(
        "Repository checkout is not current with GitHub main. "
        "Choose Runtime > Restart session, then Run all. "
        f"Local={local_commit[:12]} GitHub={remote_commit[:12]}"
    )

required_current_files = [
    "src/training/baseline_pipeline.py",
    "src/training/ecg_fm_pipeline.py",
    "src/evaluation/baseline_matrix.py",
    "src/evaluation/ecg_fm_matrix.py",
    "src/evaluation/composite_score.py",
    "src/models/resnet1d.py",
    "src/models/transformer1d.py",
]
missing_current_files = [
    relative for relative in required_current_files
    if not (WORKSPACE / relative).is_file()
]
if missing_current_files:
    raise RuntimeError(
        "GitHub checkout is missing required master-notebook files: "
        + ", ".join(missing_current_files)
    )

print({
    "repository_version": "CURRENT_WITH_GITHUB_MAIN",
    "commit": local_commit,
    "commit_url": (
        "https://github.com/tanushappapogu-max/"
        f"ecg-generalization-benchmark/commit/{local_commit}"
    ),
    "required_files": "PASS",
})


In [ ]:
#@title 3b. Install the uncommitted all-architecture code embedded in this notebook
import base64

EMBEDDED_FILES = {'src/models/resnet1d.py': 'IiIiUmVzaWR1YWwgMS1EIGNvbnZvbHV0aW9uYWwgYmFzZWxpbmUgZm9yIHR3ZWx2ZS1sZWFkIEVDRyByZWNvcmRpbmdzLiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBjb2xsZWN0aW9ucy5hYmMgaW1wb3J0IFNlcXVlbmNlCgppbXBvcnQgdG9yY2gKZnJvbSB0b3JjaCBpbXBvcnQgbm4KCgpjbGFzcyBSZXNpZHVhbEJsb2NrMUQobm4uTW9kdWxlKToKICAgICIiIlR3by1jb252b2x1dGlvbiByZXNpZHVhbCBibG9jayB3aXRoIG9wdGlvbmFsIHRlbXBvcmFsIGRvd25zYW1wbGluZy4iIiIKCiAgICBleHBhbnNpb24gPSAxCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGluX2NoYW5uZWxzOiBpbnQsIG91dF9jaGFubmVsczogaW50LCAqLCBzdHJpZGU6IGludCA9IDEpIC0+IE5vbmU6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgaWYgc3RyaWRlIDw9IDA6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInN0cmlkZSBtdXN0IGJlIHBvc2l0aXZlIikKICAgICAgICBzZWxmLmNvbnZvbHV0aW9ucyA9IG5uLlNlcXVlbnRpYWwoCiAgICAgICAgICAgIG5uLkNvbnYxZCgKICAgICAgICAgICAgICAgIGluX2NoYW5uZWxzLAogICAgICAgICAgICAgICAgb3V0X2NoYW5uZWxzLAogICAgICAgICAgICAgICAga2VybmVsX3NpemU9NywKICAgICAgICAgICAgICAgIHN0cmlkZT1zdHJpZGUsCiAgICAgICAgICAgICAgICBwYWRkaW5nPTMsCiAgICAgICAgICAgICAgICBiaWFzPUZhbHNlLAogICAgICAgICAgICApLAogICAgICAgICAgICBubi5CYXRjaE5vcm0xZChvdXRfY2hhbm5lbHMpLAogICAgICAgICAgICBubi5SZUxVKGlucGxhY2U9VHJ1ZSksCiAgICAgICAgICAgIG5uLkNvbnYxZCgKICAgICAgICAgICAgICAgIG91dF9jaGFubmVscywKICAgICAgICAgICAgICAgIG91dF9jaGFubmVscywKICAgICAgICAgICAgICAgIGtlcm5lbF9zaXplPTUsCiAgICAgICAgICAgICAgICBwYWRkaW5nPTIsCiAgICAgICAgICAgICAgICBiaWFzPUZhbHNlLAogICAgICAgICAgICApLAogICAgICAgICAgICBubi5CYXRjaE5vcm0xZChvdXRfY2hhbm5lbHMpLAogICAgICAgICkKICAgICAgICBzZWxmLnByb2plY3Rpb24gPSAoCiAgICAgICAgICAgIG5uLklkZW50aXR5KCkKICAgICAgICAgICAgaWYgc3RyaWRlID09IDEgYW5kIGluX2NoYW5uZWxzID09IG91dF9jaGFubmVscwogICAgICAgICAgICBlbHNlIG5uLlNlcXVlbnRpYWwoCiAgICAgICAgICAgICAgICBubi5Db252MWQoCiAgICAgICAgICAgICAgICAgICAgaW5fY2hhbm5lbHMsIG91dF9jaGFubmVscywga2VybmVsX3NpemU9MSwgc3RyaWRlPXN0cmlkZSwgYmlhcz1GYWxzZQogICAgICAgICAgICAgICAgKSwKICAgICAgICAgICAgICAgIG5uLkJhdGNoTm9ybTFkKG91dF9jaGFubmVscyksCiAgICAgICAgICAgICkKICAgICAgICApCiAgICAgICAgc2VsZi5hY3RpdmF0aW9uID0gbm4uUmVMVShpbnBsYWNlPVRydWUpCgogICAgZGVmIGZvcndhcmQoc2VsZiwgc2lnbmFsOiB0b3JjaC5UZW5zb3IpIC0+IHRvcmNoLlRlbnNvcjoKICAgICAgICByZXR1cm4gc2VsZi5hY3RpdmF0aW9uKHNlbGYuY29udm9sdXRpb25zKHNpZ25hbCkgKyBzZWxmLnByb2plY3Rpb24oc2lnbmFsKSkKCgpjbGFzcyBSZXNOZXQxRChubi5Nb2R1bGUpOgogICAgIiIiRm91ci1zdGFnZSBSZXNOZXQgYWRhcHRlZCB0byBmaXhlZCB0d2VsdmUtbGVhZCwgNTAwIEh6IEVDRyBpbnB1dC4iIiIKCiAgICBkZWYgX19pbml0X18oCiAgICAgICAgc2VsZiwKICAgICAgICAqLAogICAgICAgIGluX2NoYW5uZWxzOiBpbnQgPSAxMiwKICAgICAgICBudW1fb3V0cHV0czogaW50ID0gNSwKICAgICAgICBiYXNlX2NoYW5uZWxzOiBpbnQgPSAzMiwKICAgICAgICBibG9ja3NfcGVyX3N0YWdlOiBTZXF1ZW5jZVtpbnRdID0gKDIsIDIsIDIsIDIpLAogICAgICAgIGRyb3BvdXQ6IGZsb2F0ID0gMC4wLAogICAgKSAtPiBOb25lOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIGlmIGluX2NoYW5uZWxzIDw9IDAgb3IgbnVtX291dHB1dHMgPD0gMCBvciBiYXNlX2NoYW5uZWxzIDw9IDA6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImNoYW5uZWwgYW5kIG91dHB1dCBjb3VudHMgbXVzdCBiZSBwb3NpdGl2ZSIpCiAgICAgICAgaWYgbGVuKGJsb2Nrc19wZXJfc3RhZ2UpICE9IDQgb3IgYW55KGludCh2YWx1ZSkgPD0gMCBmb3IgdmFsdWUgaW4gYmxvY2tzX3Blcl9zdGFnZSk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImJsb2Nrc19wZXJfc3RhZ2UgbXVzdCBjb250YWluIGZvdXIgcG9zaXRpdmUgaW50ZWdlcnMiKQoKICAgICAgICBzZWxmLnN0ZW0gPSBubi5TZXF1ZW50aWFsKAogICAgICAgICAgICBubi5Db252MWQoCiAgICAgICAgICAgICAgICBpbl9jaGFubmVscywKICAgICAgICAgICAgICAgIGJhc2VfY2hhbm5lbHMsCiAgICAgICAgICAgICAgICBrZXJuZWxfc2l6ZT0xNSwKICAgICAgICAgICAgICAgIHN0cmlkZT0yLAogICAgICAgICAgICAgICAgcGFkZGluZz03LAogICAgICAgICAgICAgICAgYmlhcz1GYWxzZSwKICAgICAgICAgICAgKSwKICAgICAgICAgICAgbm4uQmF0Y2hOb3JtMWQoYmFzZV9jaGFubmVscyksCiAgICAgICAgICAgIG5uLlJlTFUoaW5wbGFjZT1UcnVlKSwKICAgICAgICAgICAgbm4uTWF4UG9vbDFkKGtlcm5lbF9zaXplPTMsIHN0cmlkZT0yLCBwYWRkaW5nPTEpLAogICAgICAgICkKICAgICAgICBzdGFnZXM6IGxpc3Rbbm4uTW9kdWxlXSA9IFtdCiAgICAgICAgY3VycmVudF9jaGFubmVscyA9IGJhc2VfY2hhbm5lbHMKICAgICAgICBmb3Igc3RhZ2VfaW5kZXgsIGJsb2NrX2NvdW50IGluIGVudW1lcmF0ZShibG9ja3NfcGVyX3N0YWdlKToKICAgICAgICAgICAgb3V0cHV0X2NoYW5uZWxzID0gYmFzZV9jaGFubmVscyAqICgyKipzdGFnZV9pbmRleCkKICAgICAgICAgICAgYmxvY2tzOiBsaXN0W25uLk1vZHVsZV0gPSBbXQogICAgICAgICAgICBmb3IgYmxvY2tfaW5kZXggaW4gcmFuZ2UoaW50KGJsb2NrX2NvdW50KSk6CiAgICAgICAgICAgICAgICBzdHJpZGUgPSAyIGlmIHN0YWdlX2luZGV4ID4gMCBhbmQgYmxvY2tfaW5kZXggPT0gMCBlbHNlIDEKICAgICAgICAgICAgICAgIGJsb2Nrcy5hcHBlbmQoCiAgICAgICAgICAgICAgICAgICAgUmVzaWR1YWxCbG9jazFEKAogICAgICAgICAgICAgICAgICAgICAgICBjdXJyZW50X2NoYW5uZWxzLCBvdXRwdXRfY2hhbm5lbHMsIHN0cmlkZT1zdHJpZGUKICAgICAgICAgICAgICAgICAgICApCiAgICAgICAgICAgICAgICApCiAgICAgICAgICAgICAgICBjdXJyZW50X2NoYW5uZWxzID0gb3V0cHV0X2NoYW5uZWxzCiAgICAgICAgICAgIHN0YWdlcy5hcHBlbmQobm4uU2VxdWVudGlhbCgqYmxvY2tzKSkKICAgICAgICBzZWxmLnN0YWdlcyA9IG5uLlNlcXVlbnRpYWwoKnN0YWdlcykKICAgICAgICBzZWxmLnBvb2wgPSBubi5BZGFwdGl2ZUF2Z1Bvb2wxZCgxKQogICAgICAgIHNlbGYuZHJvcG91dCA9IG5uLkRyb3BvdXQoZHJvcG91dCkKICAgICAgICBzZWxmLmNsYXNzaWZpZXIgPSBubi5MaW5lYXIoY3VycmVudF9jaGFubmVscywgbnVtX291dHB1dHMpCgogICAgICAgIGZvciBtb2R1bGUgaW4gc2VsZi5tb2R1bGVzKCk6CiAgICAgICAgICAgIGlmIGlzaW5zdGFuY2UobW9kdWxlLCBubi5Db252MWQpOgogICAgICAgICAgICAgICAgbm4uaW5pdC5rYWltaW5nX25vcm1hbF8obW9kdWxlLndlaWdodCwgbW9kZT0iZmFuX291dCIsIG5vbmxpbmVhcml0eT0icmVsdSIpCiAgICAgICAgICAgIGVsaWYgaXNpbnN0YW5jZShtb2R1bGUsIG5uLkJhdGNoTm9ybTFkKToKICAgICAgICAgICAgICAgIG5uLmluaXQub25lc18obW9kdWxlLndlaWdodCkKICAgICAgICAgICAgICAgIG5uLmluaXQuemVyb3NfKG1vZHVsZS5iaWFzKQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIHNpZ25hbDogdG9yY2guVGVuc29yKSAtPiB0b3JjaC5UZW5zb3I6CiAgICAgICAgaWYgc2lnbmFsLm5kaW0gIT0gMyBvciBzaWduYWwuc2hhcGVbMV0gIT0gMTI6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoCiAgICAgICAgICAgICAgICBmIkV4cGVjdGVkIHNpZ25hbCBzaGFwZWQgKGJhdGNoLCAxMiwgc2FtcGxlcyksIGdvdCB7dHVwbGUoc2lnbmFsLnNoYXBlKX0iCiAgICAgICAgICAgICkKICAgICAgICBmZWF0dXJlcyA9IHNlbGYuc3RhZ2VzKHNlbGYuc3RlbShzaWduYWwpKQogICAgICAgIHBvb2xlZCA9IHNlbGYucG9vbChmZWF0dXJlcykuc3F1ZWV6ZSgtMSkKICAgICAgICByZXR1cm4gc2VsZi5jbGFzc2lmaWVyKHNlbGYuZHJvcG91dChwb29sZWQpKQoK', 'src/models/transformer1d.py': 'IiIiVmFuaWxsYSBwYXRjaCBUcmFuc2Zvcm1lciBiYXNlbGluZSBmb3IgdHdlbHZlLWxlYWQgRUNHIHJlY29yZGluZ3MuIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgdG9yY2gKZnJvbSB0b3JjaCBpbXBvcnQgbm4KCgpjbGFzcyBFQ0dUcmFuc2Zvcm1lcjFEKG5uLk1vZHVsZSk6CiAgICAiIiJQYXRjaCBlbWJlZGRpbmcgKyBwb3NpdGlvbmFsIGVuY29kaW5nICsgVHJhbnNmb3JtZXIgZW5jb2RlciBiYXNlbGluZS4iIiIKCiAgICBkZWYgX19pbml0X18oCiAgICAgICAgc2VsZiwKICAgICAgICAqLAogICAgICAgIGluX2NoYW5uZWxzOiBpbnQgPSAxMiwKICAgICAgICBudW1fb3V0cHV0czogaW50ID0gNSwKICAgICAgICBpbnB1dF9zYW1wbGVzOiBpbnQgPSA1MDAwLAogICAgICAgIHBhdGNoX3NpemU6IGludCA9IDUwLAogICAgICAgIGVtYmVkX2RpbTogaW50ID0gMTI4LAogICAgICAgIG51bV9oZWFkczogaW50ID0gNCwKICAgICAgICBudW1fbGF5ZXJzOiBpbnQgPSA0LAogICAgICAgIGZlZWRmb3J3YXJkX2RpbTogaW50ID0gMjU2LAogICAgICAgIGRyb3BvdXQ6IGZsb2F0ID0gMC4xLAogICAgKSAtPiBOb25lOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIGlmIGlucHV0X3NhbXBsZXMgPD0gMCBvciBwYXRjaF9zaXplIDw9IDAgb3IgaW5wdXRfc2FtcGxlcyAlIHBhdGNoX3NpemU6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoImlucHV0X3NhbXBsZXMgbXVzdCBiZSBhIHBvc2l0aXZlIG11bHRpcGxlIG9mIHBhdGNoX3NpemUiKQogICAgICAgIGlmIGVtYmVkX2RpbSA8PSAwIG9yIG51bV9oZWFkcyA8PSAwIG9yIGVtYmVkX2RpbSAlIG51bV9oZWFkczoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiZW1iZWRfZGltIG11c3QgYmUgcG9zaXRpdmUgYW5kIGRpdmlzaWJsZSBieSBudW1faGVhZHMiKQogICAgICAgIGlmIG51bV9sYXllcnMgPD0gMCBvciBmZWVkZm9yd2FyZF9kaW0gPD0gMCBvciBudW1fb3V0cHV0cyA8PSAwOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJsYXllciwgZmVlZC1mb3J3YXJkLCBhbmQgb3V0cHV0IGNvdW50cyBtdXN0IGJlIHBvc2l0aXZlIikKCiAgICAgICAgc2VsZi5pbnB1dF9zYW1wbGVzID0gaW50KGlucHV0X3NhbXBsZXMpCiAgICAgICAgc2VsZi5wYXRjaF9zaXplID0gaW50KHBhdGNoX3NpemUpCiAgICAgICAgcGF0Y2hfY291bnQgPSBpbnB1dF9zYW1wbGVzIC8vIHBhdGNoX3NpemUKICAgICAgICBzZWxmLnBhdGNoX2VtYmVkZGluZyA9IG5uLkNvbnYxZCgKICAgICAgICAgICAgaW5fY2hhbm5lbHMsCiAgICAgICAgICAgIGVtYmVkX2RpbSwKICAgICAgICAgICAga2VybmVsX3NpemU9cGF0Y2hfc2l6ZSwKICAgICAgICAgICAgc3RyaWRlPXBhdGNoX3NpemUsCiAgICAgICAgICAgIGJpYXM9VHJ1ZSwKICAgICAgICApCiAgICAgICAgc2VsZi5jbGFzc190b2tlbiA9IG5uLlBhcmFtZXRlcih0b3JjaC56ZXJvcygxLCAxLCBlbWJlZF9kaW0pKQogICAgICAgIHNlbGYucG9zaXRpb25fZW1iZWRkaW5nID0gbm4uUGFyYW1ldGVyKAogICAgICAgICAgICB0b3JjaC56ZXJvcygxLCBwYXRjaF9jb3VudCArIDEsIGVtYmVkX2RpbSkKICAgICAgICApCiAgICAgICAgbGF5ZXIgPSBubi5UcmFuc2Zvcm1lckVuY29kZXJMYXllcigKICAgICAgICAgICAgZF9tb2RlbD1lbWJlZF9kaW0sCiAgICAgICAgICAgIG5oZWFkPW51bV9oZWFkcywKICAgICAgICAgICAgZGltX2ZlZWRmb3J3YXJkPWZlZWRmb3J3YXJkX2RpbSwKICAgICAgICAgICAgZHJvcG91dD1kcm9wb3V0LAogICAgICAgICAgICBhY3RpdmF0aW9uPSJnZWx1IiwKICAgICAgICAgICAgYmF0Y2hfZmlyc3Q9VHJ1ZSwKICAgICAgICAgICAgbm9ybV9maXJzdD1UcnVlLAogICAgICAgICkKICAgICAgICBzZWxmLmVuY29kZXIgPSBubi5UcmFuc2Zvcm1lckVuY29kZXIoCiAgICAgICAgICAgIGxheWVyLCBudW1fbGF5ZXJzPW51bV9sYXllcnMsIGVuYWJsZV9uZXN0ZWRfdGVuc29yPUZhbHNlCiAgICAgICAgKQogICAgICAgIHNlbGYubm9ybSA9IG5uLkxheWVyTm9ybShlbWJlZF9kaW0pCiAgICAgICAgc2VsZi5kcm9wb3V0ID0gbm4uRHJvcG91dChkcm9wb3V0KQogICAgICAgIHNlbGYuY2xhc3NpZmllciA9IG5uLkxpbmVhcihlbWJlZF9kaW0sIG51bV9vdXRwdXRzKQoKICAgICAgICBubi5pbml0LnRydW5jX25vcm1hbF8oc2VsZi5jbGFzc190b2tlbiwgc3RkPTAuMDIpCiAgICAgICAgbm4uaW5pdC50cnVuY19ub3JtYWxfKHNlbGYucG9zaXRpb25fZW1iZWRkaW5nLCBzdGQ9MC4wMikKICAgICAgICBubi5pbml0Lnhhdmllcl91bmlmb3JtXyhzZWxmLnBhdGNoX2VtYmVkZGluZy53ZWlnaHQpCiAgICAgICAgbm4uaW5pdC56ZXJvc18oc2VsZi5wYXRjaF9lbWJlZGRpbmcuYmlhcykKICAgICAgICBubi5pbml0Lnhhdmllcl91bmlmb3JtXyhzZWxmLmNsYXNzaWZpZXIud2VpZ2h0KQogICAgICAgIG5uLmluaXQuemVyb3NfKHNlbGYuY2xhc3NpZmllci5iaWFzKQoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIHNpZ25hbDogdG9yY2guVGVuc29yKSAtPiB0b3JjaC5UZW5zb3I6CiAgICAgICAgaWYgc2lnbmFsLm5kaW0gIT0gMyBvciBzaWduYWwuc2hhcGVbMV0gIT0gMTI6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoCiAgICAgICAgICAgICAgICBmIkV4cGVjdGVkIHNpZ25hbCBzaGFwZWQgKGJhdGNoLCAxMiwgc2FtcGxlcyksIGdvdCB7dHVwbGUoc2lnbmFsLnNoYXBlKX0iCiAgICAgICAgICAgICkKICAgICAgICBpZiBzaWduYWwuc2hhcGVbMl0gIT0gc2VsZi5pbnB1dF9zYW1wbGVzOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKAogICAgICAgICAgICAgICAgZiJFeHBlY3RlZCB7c2VsZi5pbnB1dF9zYW1wbGVzfSBzYW1wbGVzLCBnb3Qge3NpZ25hbC5zaGFwZVsyXX0iCiAgICAgICAgICAgICkKICAgICAgICBwYXRjaGVzID0gc2VsZi5wYXRjaF9lbWJlZGRpbmcoc2lnbmFsKS50cmFuc3Bvc2UoMSwgMikKICAgICAgICBjbGFzc190b2tlbiA9IHNlbGYuY2xhc3NfdG9rZW4uZXhwYW5kKGxlbihzaWduYWwpLCAtMSwgLTEpCiAgICAgICAgdG9rZW5zID0gdG9yY2guY2F0KChjbGFzc190b2tlbiwgcGF0Y2hlcyksIGRpbT0xKQogICAgICAgIHRva2VucyA9IHRva2VucyArIHNlbGYucG9zaXRpb25fZW1iZWRkaW5nWzosIDogdG9rZW5zLnNoYXBlWzFdXQogICAgICAgIGVuY29kZWQgPSBzZWxmLmVuY29kZXIodG9rZW5zKQogICAgICAgIHJldHVybiBzZWxmLmNsYXNzaWZpZXIoc2VsZi5kcm9wb3V0KHNlbGYubm9ybShlbmNvZGVkWzosIDBdKSkpCgo=', 'src/training/baseline_pipeline.py': '...', 'src/evaluation/baseline_matrix.py': 'IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJFdmFsdWF0ZSBmcm9tLXNjcmF0Y2ggYmFzZWxpbmUgY2hlY2twb2ludHMgYWNyb3NzIGV2ZXJ5IHNvdXJjZS10YXJnZXQgcGFpci4iIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBhcmdwYXJzZQppbXBvcnQganNvbgppbXBvcnQgbWF0aApmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKZnJvbSB0eXBpbmcgaW1wb3J0IEFueSwgU2VxdWVuY2UKCmltcG9ydCBwYW5kYXMgYXMgcGQKaW1wb3J0IHRvcmNoCmZyb20gdG9yY2ggaW1wb3J0IG5uCgp0cnk6CiAgICBmcm9tIHNyYy5kYXRhLndlZWsyX21hbmlmZXN0IGltcG9ydCB2YWxpZGF0ZV9jYW5vbmljYWxfbWFuaWZlc3QKICAgIGZyb20gc3JjLmV2YWx1YXRpb24uZWNnX2ZtX21hdHJpeCBpbXBvcnQgKAogICAgICAgIERFRkFVTFRfREFUQVNFVFMsCiAgICAgICAgbWF0cml4X2Zyb21fbG9uZywKICAgICAgICBwYXJzZV9uYW1lZF9wYXRocywKICAgICkKICAgIGZyb20gc3JjLnRyYWluaW5nLmJhc2VsaW5lX3BpcGVsaW5lIGltcG9ydCBBUkNISVRFQ1RVUkVTLCBidWlsZF9iYXNlbGluZV9tb2RlbAogICAgZnJvbSBzcmMudHJhaW5pbmcuZWNnX2ZtX3BpcGVsaW5lIGltcG9ydCAoCiAgICAgICAgQ0xBU1NfTkFNRVMsCiAgICAgICAgRUNHTWFuaWZlc3REYXRhc2V0LAogICAgICAgIExBQkVMX0NPTFVNTlMsCiAgICAgICAgX2xvYWRlciwKICAgICAgICBldmFsdWF0ZSwKICAgICAgICBzZWVkX2V2ZXJ5dGhpbmcsCiAgICApCmV4Y2VwdCBNb2R1bGVOb3RGb3VuZEVycm9yOgogICAgaW1wb3J0IHN5cwoKICAgIHN5cy5wYXRoLmluc2VydCgwLCBzdHIoUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudHNbMl0pKQogICAgZnJvbSBzcmMuZGF0YS53ZWVrMl9tYW5pZmVzdCBpbXBvcnQgdmFsaWRhdGVfY2Fub25pY2FsX21hbmlmZXN0CiAgICBmcm9tIHNyYy5ldmFsdWF0aW9uLmVjZ19mbV9tYXRyaXggaW1wb3J0ICgKICAgICAgICBERUZBVUxUX0RBVEFTRVRTLAogICAgICAgIG1hdHJpeF9mcm9tX2xvbmcsCiAgICAgICAgcGFyc2VfbmFtZWRfcGF0aHMsCiAgICApCiAgICBmcm9tIHNyYy50cmFpbmluZy5iYXNlbGluZV9waXBlbGluZSBpbXBvcnQgQVJDSElURUNUVVJFUywgYnVpbGRfYmFzZWxpbmVfbW9kZWwKICAgIGZyb20gc3JjLnRyYWluaW5nLmVjZ19mbV9waXBlbGluZSBpbXBvcnQgKAogICAgICAgIENMQVNTX05BTUVTLAogICAgICAgIEVDR01hbmlmZXN0RGF0YXNldCwKICAgICAgICBMQUJFTF9DT0xVTU5TLAogICAgICAgIF9sb2FkZXIsCiAgICAgICAgZXZhbHVhdGUsCiAgICAgICAgc2VlZF9ldmVyeXRoaW5nLAogICAgKQoKCmRlZiBfd3JpdGVfcHJlZGljdGlvbnMocGF0aDogUGF0aCwgcmVzdWx0OiBkaWN0W3N0ciwgQW55XSkgLT4gTm9uZToKICAgIHZhbHVlczogZGljdFtzdHIsIEFueV0gPSB7InJlY29yZF9pZCI6IHJlc3VsdFsicmVjb3JkX2lkcyJdfQogICAgZm9yIGluZGV4LCBsYWJlbCBpbiBlbnVtZXJhdGUoTEFCRUxfQ09MVU1OUyk6CiAgICAgICAgdmFsdWVzW2YidGFyZ2V0X3tsYWJlbH0iXSA9IHJlc3VsdFsieV90cnVlIl1bOiwgaW5kZXhdLmFzdHlwZShpbnQpCiAgICAgICAgdmFsdWVzW2YicHJvYmFiaWxpdHlfe2xhYmVsfSJdID0gcmVzdWx0WyJ5X3Njb3JlIl1bOiwgaW5kZXhdCiAgICBwYXRoLnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBwZC5EYXRhRnJhbWUodmFsdWVzKS50b19jc3YocGF0aCwgaW5kZXg9RmFsc2UpCgoKZGVmIHJ1bl9tYXRyaXgoYXJnczogYXJncGFyc2UuTmFtZXNwYWNlKSAtPiBwZC5EYXRhRnJhbWU6CiAgICBzZWVkX2V2ZXJ5dGhpbmcoYXJncy5zZWVkKQogICAgZGF0YXNldHMgPSB0dXBsZShhcmdzLmRhdGFzZXRzKQogICAgaWYgbGVuKHNldChkYXRhc2V0cykpICE9IGxlbihkYXRhc2V0cyk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiLS1kYXRhc2V0cyBjb250YWlucyBkdXBsaWNhdGVzIikKICAgIHNpZ25hbF9yb290cyA9IHBhcnNlX25hbWVkX3BhdGhzKGFyZ3Muc2lnbmFsX3Jvb3QpCiAgICBkZXZpY2UgPSB0b3JjaC5kZXZpY2UoCiAgICAgICAgYXJncy5kZXZpY2UKICAgICAgICBpZiBhcmdzLmRldmljZSAhPSAiYXV0byIKICAgICAgICBlbHNlICgiY3VkYSIgaWYgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKSBlbHNlICJjcHUiKQogICAgKQogICAgaWYgZGV2aWNlLnR5cGUgPT0gImN1ZGEiIGFuZCBub3QgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIkNVREEgd2FzIHJlcXVlc3RlZCBidXQgaXMgdW5hdmFpbGFibGUiKQogICAgdXNlX2FtcCA9IGJvb2woYXJncy5taXhlZF9wcmVjaXNpb24gYW5kIGRldmljZS50eXBlID09ICJjdWRhIikKICAgIGFyZ3Mub3V0cHV0X2Rpci5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCgogICAgdGFyZ2V0czogZGljdFtzdHIsIEVDR01hbmlmZXN0RGF0YXNldF0gPSB7fQogICAgdGFyZ2V0X2Vycm9yczogZGljdFtzdHIsIHN0cl0gPSB7fQogICAgZm9yIHRhcmdldCBpbiBkYXRhc2V0czoKICAgICAgICBtYW5pZmVzdF9wYXRoID0gYXJncy5tYW5pZmVzdF9yb290IC8gZiJ7dGFyZ2V0fV93ZWVrMi5jc3YiCiAgICAgICAgc2lnbmFsX3Jvb3QgPSBzaWduYWxfcm9vdHMuZ2V0KHRhcmdldCkKICAgICAgICBpZiBub3QgbWFuaWZlc3RfcGF0aC5pc19maWxlKCk6CiAgICAgICAgICAgIHRhcmdldF9lcnJvcnNbdGFyZ2V0XSA9IGYibWlzc2luZyBtYW5pZmVzdDoge21hbmlmZXN0X3BhdGh9IgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGlmIHNpZ25hbF9yb290IGlzIE5vbmUgb3Igbm90IHNpZ25hbF9yb290LmV4aXN0cygpOgogICAgICAgICAgICB0YXJnZXRfZXJyb3JzW3RhcmdldF0gPSBmIm1pc3Npbmcgc2lnbmFsIHJvb3QgZm9yIHt0YXJnZXR9IgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIHRyeToKICAgICAgICAgICAgbWFuaWZlc3QgPSB2YWxpZGF0ZV9jYW5vbmljYWxfbWFuaWZlc3QoCiAgICAgICAgICAgICAgICBwZC5yZWFkX2NzdihtYW5pZmVzdF9wYXRoLCBsb3dfbWVtb3J5PUZhbHNlKQogICAgICAgICAgICApCiAgICAgICAgICAgIHRhcmdldHNbdGFyZ2V0XSA9IEVDR01hbmlmZXN0RGF0YXNldCgKICAgICAgICAgICAgICAgIG1hbmlmZXN0LAogICAgICAgICAgICAgICAgc3BsaXQ9InRlc3QiLAogICAgICAgICAgICAgICAgc2lnbmFsX3Jvb3Q9c2lnbmFsX3Jvb3QsCiAgICAgICAgICAgICAgICBtYXhfcmVjb3Jkcz1hcmdzLm1heF9yZWNvcmRzX3Blcl90YXJnZXQsCiAgICAgICAgICAgICkKICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGV4YzoKICAgICAgICAgICAgdGFyZ2V0X2Vycm9yc1t0YXJnZXRdID0gZiJpbnZhbGlkIHRhcmdldCBkYXRhOiB7ZXhjfSIKCiAgICByb3dzOiBsaXN0W2RpY3Rbc3RyLCBBbnldXSA9IFtdCiAgICBmb3Igc291cmNlIGluIGRhdGFzZXRzOgogICAgICAgIGNoZWNrcG9pbnRfcGF0aCA9IGFyZ3Muc291cmNlX3J1bnNfcm9vdCAvIHNvdXJjZSAvICJiZXN0X2NoZWNrcG9pbnQucHQiCiAgICAgICAgaWYgbm90IGNoZWNrcG9pbnRfcGF0aC5pc19maWxlKCk6CiAgICAgICAgICAgIGZvciB0YXJnZXQgaW4gZGF0YXNldHM6CiAgICAgICAgICAgICAgICByb3dzLmFwcGVuZCgKICAgICAgICAgICAgICAgICAgICB7CiAgICAgICAgICAgICAgICAgICAgICAgICJhcmNoaXRlY3R1cmUiOiBhcmdzLmFyY2hpdGVjdHVyZSwKICAgICAgICAgICAgICAgICAgICAgICAgInRhc2siOiAiZml2ZV9sYWJlbCIsCiAgICAgICAgICAgICAgICAgICAgICAgICJzb3VyY2VfZGF0YXNldCI6IHNvdXJjZSwKICAgICAgICAgICAgICAgICAgICAgICAgInRhcmdldF9kYXRhc2V0IjogdGFyZ2V0LAogICAgICAgICAgICAgICAgICAgICAgICAic3RhdHVzIjogIkJMT0NLRURfTUlTU0lOR19TT1VSQ0VfQ0hFQ0tQT0lOVCIsCiAgICAgICAgICAgICAgICAgICAgICAgICJkZXRhaWwiOiBzdHIoY2hlY2twb2ludF9wYXRoKSwKICAgICAgICAgICAgICAgICAgICAgICAgInJlY29yZF9jb3VudCI6IDAsCiAgICAgICAgICAgICAgICAgICAgICAgICJtYWNyb19hdXJvYyI6IG1hdGgubmFuLAogICAgICAgICAgICAgICAgICAgICAgICAqKntmImF1cm9jX3tuYW1lfSI6IG1hdGgubmFuIGZvciBuYW1lIGluIENMQVNTX05BTUVTfSwKICAgICAgICAgICAgICAgICAgICB9CiAgICAgICAgICAgICAgICApCiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgY2hlY2twb2ludCA9IHRvcmNoLmxvYWQoY2hlY2twb2ludF9wYXRoLCBtYXBfbG9jYXRpb249ZGV2aWNlLCB3ZWlnaHRzX29ubHk9RmFsc2UpCiAgICAgICAgaWYgY2hlY2twb2ludC5nZXQoImFyY2hpdGVjdHVyZSIpICE9IGFyZ3MuYXJjaGl0ZWN0dXJlOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKAogICAgICAgICAgICAgICAgZiJDaGVja3BvaW50IGFyY2hpdGVjdHVyZSBtaXNtYXRjaDoge2NoZWNrcG9pbnQuZ2V0KCdhcmNoaXRlY3R1cmUnKSFyfSIKICAgICAgICAgICAgKQogICAgICAgIGlmIGNoZWNrcG9pbnQuZ2V0KCJkYXRhc2V0IikgIT0gc291cmNlOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKAogICAgICAgICAgICAgICAgZiJDaGVja3BvaW50IGRhdGFzZXQgbWlzbWF0Y2g6IHtjaGVja3BvaW50LmdldCgnZGF0YXNldCcpIXJ9IgogICAgICAgICAgICApCiAgICAgICAgcnVuX2NvbmZpZyA9IGNoZWNrcG9pbnQuZ2V0KCJydW5fY29uZmlnIiwge30pCiAgICAgICAgbW9kZWwsIF8gPSBidWlsZF9iYXNlbGluZV9tb2RlbCgKICAgICAgICAgICAgYXJncy5hcmNoaXRlY3R1cmUsCiAgICAgICAgICAgIG1vZGVsX2NvbmZpZz1kaWN0KHJ1bl9jb25maWcuZ2V0KCJtb2RlbF9jb25maWciLCB7fSkpLAogICAgICAgICAgICBudW1fb3V0cHV0cz1sZW4oTEFCRUxfQ09MVU1OUyksCiAgICAgICAgKQogICAgICAgIG1vZGVsLmxvYWRfc3RhdGVfZGljdChjaGVja3BvaW50WyJtb2RlbF9zdGF0ZV9kaWN0Il0pCiAgICAgICAgbW9kZWwudG8oZGV2aWNlKQogICAgICAgIG1vZGVsLmV2YWwoKQoKICAgICAgICBmb3IgdGFyZ2V0IGluIGRhdGFzZXRzOgogICAgICAgICAgICBpZiB0YXJnZXQgaW4gdGFyZ2V0X2Vycm9yczoKICAgICAgICAgICAgICAgIHJvd3MuYXBwZW5kKAogICAgICAgICAgICAgICAgICAgIHsKICAgICAgICAgICAgICAgICAgICAgICAgImFyY2hpdGVjdHVyZSI6IGFyZ3MuYXJjaGl0ZWN0dXJlLAogICAgICAgICAgICAgICAgICAgICAgICAidGFzayI6ICJmaXZlX2xhYmVsIiwKICAgICAgICAgICAgICAgICAgICAgICAgInNvdXJjZV9kYXRhc2V0Ijogc291cmNlLAogICAgICAgICAgICAgICAgICAgICAgICAidGFyZ2V0X2RhdGFzZXQiOiB0YXJnZXQsCiAgICAgICAgICAgICAgICAgICAgICAgICJzdGF0dXMiOiAiQkxPQ0tFRF9NSVNTSU5HX09SX0lOVkFMSURfVEFSR0VUX0RBVEEiLAogICAgICAgICAgICAgICAgICAgICAgICAiZGV0YWlsIjogdGFyZ2V0X2Vycm9yc1t0YXJnZXRdLAogICAgICAgICAgICAgICAgICAgICAgICAicmVjb3JkX2NvdW50IjogMCwKICAgICAgICAgICAgICAgICAgICAgICAgIm1hY3JvX2F1cm9jIjogbWF0aC5uYW4sCiAgICAgICAgICAgICAgICAgICAgICAgICoqe2YiYXVyb2Nfe25hbWV9IjogbWF0aC5uYW4gZm9yIG5hbWUgaW4gQ0xBU1NfTkFNRVN9LAogICAgICAgICAgICAgICAgICAgIH0KICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgIGxvYWRlciA9IF9sb2FkZXIoCiAgICAgICAgICAgICAgICB0YXJnZXRzW3RhcmdldF0sCiAgICAgICAgICAgICAgICBiYXRjaF9zaXplPWFyZ3MuYmF0Y2hfc2l6ZSwKICAgICAgICAgICAgICAgIHNodWZmbGU9RmFsc2UsCiAgICAgICAgICAgICAgICB3b3JrZXJzPWFyZ3MubnVtX3dvcmtlcnMsCiAgICAgICAgICAgICAgICBzZWVkPWFyZ3Muc2VlZCwKICAgICAgICAgICAgKQogICAgICAgICAgICByZXN1bHQgPSBldmFsdWF0ZSgKICAgICAgICAgICAgICAgIG1vZGVsLAogICAgICAgICAgICAgICAgbG9hZGVyLAogICAgICAgICAgICAgICAgZGV2aWNlPWRldmljZSwKICAgICAgICAgICAgICAgIGNyaXRlcmlvbj1ubi5CQ0VXaXRoTG9naXRzTG9zcygpLAogICAgICAgICAgICAgICAgdXNlX2FtcD11c2VfYW1wLAogICAgICAgICAgICApCiAgICAgICAgICAgIGNlbGxfZGlyID0gYXJncy5vdXRwdXRfZGlyIC8gInByZWRpY3Rpb25zIiAvIGYie3NvdXJjZX1fX3RvX197dGFyZ2V0fSIKICAgICAgICAgICAgX3dyaXRlX3ByZWRpY3Rpb25zKGNlbGxfZGlyIC8gInRlc3RfcHJlZGljdGlvbnMuY3N2IiwgcmVzdWx0KQogICAgICAgICAgICBtZXRyaWNzID0gewogICAgICAgICAgICAgICAgImFyY2hpdGVjdHVyZSI6IGFyZ3MuYXJjaGl0ZWN0dXJlLAogICAgICAgICAgICAgICAgInRhc2siOiAiZml2ZV9sYWJlbCIsCiAgICAgICAgICAgICAgICAic291cmNlX2RhdGFzZXQiOiBzb3VyY2UsCiAgICAgICAgICAgICAgICAidGFyZ2V0X2RhdGFzZXQiOiB0YXJnZXQsCiAgICAgICAgICAgICAgICAic3RhdHVzIjogIkNPTVBMRVRFIiwKICAgICAgICAgICAgICAgICJkZXRhaWwiOiAiIiwKICAgICAgICAgICAgICAgICJyZWNvcmRfY291bnQiOiBsZW4ocmVzdWx0WyJyZWNvcmRfaWRzIl0pLAogICAgICAgICAgICAgICAgIm1hY3JvX2F1cm9jIjogcmVzdWx0WyJtYWNyb19hdXJvYyJdLAogICAgICAgICAgICAgICAgKip7CiAgICAgICAgICAgICAgICAgICAgZiJhdXJvY197bmFtZX0iOiByZXN1bHRbInBlcl9jbGFzc19hdXJvYyJdW25hbWVdCiAgICAgICAgICAgICAgICAgICAgZm9yIG5hbWUgaW4gQ0xBU1NfTkFNRVMKICAgICAgICAgICAgICAgIH0sCiAgICAgICAgICAgIH0KICAgICAgICAgICAgKGNlbGxfZGlyIC8gIm1ldHJpY3MuanNvbiIpLndyaXRlX3RleHQoCiAgICAgICAgICAgICAgICBqc29uLmR1bXBzKG1ldHJpY3MsIGluZGVudD0yLCBhbGxvd19uYW49VHJ1ZSkgKyAiXG4iLAogICAgICAgICAgICAgICAgZW5jb2Rpbmc9InV0Zi04IiwKICAgICAgICAgICAgKQogICAgICAgICAgICByb3dzLmFwcGVuZChtZXRyaWNzKQogICAgICAgIGRlbCBtb2RlbAogICAgICAgIGlmIGRldmljZS50eXBlID09ICJjdWRhIjoKICAgICAgICAgICAgdG9yY2guY3VkYS5lbXB0eV9jYWNoZSgpCgogICAgcmVzdWx0cyA9IHBkLkRhdGFGcmFtZShyb3dzKQogICAgZXhwZWN0ZWQgPSBsZW4oZGF0YXNldHMpICoqIDIKICAgIGlmIGxlbihyZXN1bHRzKSAhPSBleHBlY3RlZDoKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJFeHBlY3RlZCB7ZXhwZWN0ZWR9IG1hdHJpeCByb3dzLCBwcm9kdWNlZCB7bGVuKHJlc3VsdHMpfSIpCiAgICBsb25nX3BhdGggPSBhcmdzLm91dHB1dF9kaXIgLyBmInthcmdzLmFyY2hpdGVjdHVyZX1fZml2ZV9sYWJlbF9tYXRyaXhfbG9uZy5jc3YiCiAgICByZXN1bHRzLnRvX2Nzdihsb25nX3BhdGgsIGluZGV4PUZhbHNlKQogICAgbWF0cml4X2Zyb21fbG9uZyhyZXN1bHRzLCBkYXRhc2V0cykudG9fY3N2KAogICAgICAgIGFyZ3Mub3V0cHV0X2RpciAvIGYie2FyZ3MuYXJjaGl0ZWN0dXJlfV9maXZlX2xhYmVsX21hY3JvX2F1cm9jX21hdHJpeC5jc3YiCiAgICApCiAgICBzdW1tYXJ5ID0gewogICAgICAgICJhcmNoaXRlY3R1cmUiOiBhcmdzLmFyY2hpdGVjdHVyZSwKICAgICAgICAidGFzayI6ICJmaXZlX2xhYmVsIiwKICAgICAgICAiZXhwZWN0ZWRfY2VsbHMiOiBleHBlY3RlZCwKICAgICAgICAiY29tcGxldGVkX2NlbGxzIjogaW50KHJlc3VsdHNbInN0YXR1cyJdLmVxKCJDT01QTEVURSIpLnN1bSgpKSwKICAgICAgICAiYmxvY2tlZF9jZWxscyI6IGludChyZXN1bHRzWyJzdGF0dXMiXS5uZSgiQ09NUExFVEUiKS5zdW0oKSksCiAgICAgICAgImRhdGFzZXRzIjogbGlzdChkYXRhc2V0cyksCiAgICB9CiAgICAoYXJncy5vdXRwdXRfZGlyIC8gIm1hdHJpeF9zdW1tYXJ5Lmpzb24iKS53cml0ZV90ZXh0KAogICAgICAgIGpzb24uZHVtcHMoc3VtbWFyeSwgaW5kZW50PTIpICsgIlxuIiwgZW5jb2Rpbmc9InV0Zi04IgogICAgKQogICAgaWYgYXJncy5mYWlsX29uX21pc3NpbmcgYW5kIHN1bW1hcnlbImJsb2NrZWRfY2VsbHMiXToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJNYXRyaXggaGFzIHtzdW1tYXJ5WydibG9ja2VkX2NlbGxzJ119IGJsb2NrZWQgY2VsbHMiKQogICAgcmV0dXJuIHJlc3VsdHMKCgpkZWYgYnVpbGRfcGFyc2VyKCkgLT4gYXJncGFyc2UuQXJndW1lbnRQYXJzZXI6CiAgICBwYXJzZXIgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcihkZXNjcmlwdGlvbj1fX2RvY19fKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1hcmNoaXRlY3R1cmUiLCBjaG9pY2VzPUFSQ0hJVEVDVFVSRVMsIHJlcXVpcmVkPVRydWUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXNvdXJjZS1ydW5zLXJvb3QiLCB0eXBlPVBhdGgsIHJlcXVpcmVkPVRydWUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW1hbmlmZXN0LXJvb3QiLCB0eXBlPVBhdGgsIHJlcXVpcmVkPVRydWUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXNpZ25hbC1yb290IiwgYWN0aW9uPSJhcHBlbmQiLCBkZWZhdWx0PVtdKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1vdXRwdXQtZGlyIiwgdHlwZT1QYXRoLCByZXF1aXJlZD1UcnVlKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1kYXRhc2V0cyIsIG5hcmdzPSIrIiwgZGVmYXVsdD1saXN0KERFRkFVTFRfREFUQVNFVFMpKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1kZXZpY2UiLCBkZWZhdWx0PSJhdXRvIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tc2VlZCIsIHR5cGU9aW50LCBkZWZhdWx0PTQyKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1iYXRjaC1zaXplIiwgdHlwZT1pbnQsIGRlZmF1bHQ9MzIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW51bS13b3JrZXJzIiwgdHlwZT1pbnQsIGRlZmF1bHQ9MikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tbWF4LXJlY29yZHMtcGVyLXRhcmdldCIsIHR5cGU9aW50KQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgKICAgICAgICAiLS1taXhlZC1wcmVjaXNpb24iLCBhY3Rpb249YXJncGFyc2UuQm9vbGVhbk9wdGlvbmFsQWN0aW9uLCBkZWZhdWx0PVRydWUKICAgICkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tZmFpbC1vbi1taXNzaW5nIiwgYWN0aW9uPSJzdG9yZV90cnVlIikKICAgIHJldHVybiBwYXJzZXIKCgpkZWYgbWFpbihhcmd2OiBTZXF1ZW5jZVtzdHJdIHwgTm9uZSA9IE5vbmUpIC0+IGludDoKICAgIGFyZ3MgPSBidWlsZF9wYXJzZXIoKS5wYXJzZV9hcmdzKGFyZ3YpCiAgICByZXN1bHQgPSBydW5fbWF0cml4KGFyZ3MpCiAgICBwcmludChyZXN1bHRbWyJzb3VyY2VfZGF0YXNldCIsICJ0YXJnZXRfZGF0YXNldCIsICJzdGF0dXMiLCAibWFjcm9fYXVyb2MiXV0pCiAgICByZXR1cm4gMAoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICByYWlzZSBTeXN0ZW1FeGl0KG1haW4oKSkKCg==', 'src/evaluation/composite_score.py': 'IiIiQ29tcG9zaXRlIGNyb3NzLWRhdGFzZXQgc2NvcmUgZnJvbSB0aGUgRUNHIGJlbmNobWFyayBwcm9wb3NhbC4iIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBhcmdwYXJzZQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKZnJvbSB0eXBpbmcgaW1wb3J0IFNlcXVlbmNlCgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHBhbmRhcyBhcyBwZAoKClBBSVJfQ09MVU1OUyA9ICgic291cmNlX2RhdGFzZXQiLCAidGFyZ2V0X2RhdGFzZXQiKQpTSElGVF9DT0xVTU5TID0gKCJQUyIsICJEUyIsICJMUyIpCgoKZGVmIGNvbXB1dGVfY29tcG9zaXRlX3Njb3JlKAogICAgbWF0cml4X3Jvd3M6IHBkLkRhdGFGcmFtZSwKICAgIHNoaWZ0X3Jvd3M6IHBkLkRhdGFGcmFtZSwKICAgICosCiAgICBsYW1iZGFfcHM6IGZsb2F0ID0gMS4wLAogICAgbGFtYmRhX2RzOiBmbG9hdCA9IDIuMCwKICAgIGxhbWJkYV9sczogZmxvYXQgPSAzLjAsCikgLT4gdHVwbGVbZmxvYXQsIHBkLkRhdGFGcmFtZV06CiAgICAiIiJSZXR1cm4gcHJvcG9zYWwgU2NvcmUgYW5kIHRoZSBhdWRpdGFibGUgb2ZmLWRpYWdvbmFsIGNvbXBvbmVudCB0YWJsZS4KCiAgICBQcm9wb3NhbCBkZWZpbml0aW9uczoKICAgIGBgZGVsdGFfaWogPSBBVVJPQ19paSAtIEFVUk9DX2lqYGA7CiAgICBgYHdfaWogPSBsYW1iZGFfcHMqUFMgKyBsYW1iZGFfZHMqRFMgKyBsYW1iZGFfbHMqTFNgYDsgYW5kCiAgICBgYFNjb3JlID0gQ0QgKiAoMSAtIG1lYW4oZGVsdGFfaWogLyAod19paiArIDEpKSlgYCwgY2xpcHBlZCB0byBbMCwgMV0uCiAgICAiIiIKCiAgICBpZiBtaW4obGFtYmRhX3BzLCBsYW1iZGFfZHMsIGxhbWJkYV9scykgPCAwOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlNoaWZ0IHdlaWdodHMgbXVzdCBiZSBub24tbmVnYXRpdmUiKQogICAgcmVxdWlyZWRfbWF0cml4ID0geypQQUlSX0NPTFVNTlMsICJtYWNyb19hdXJvYyJ9CiAgICByZXF1aXJlZF9zaGlmdHMgPSB7KlBBSVJfQ09MVU1OUywgKlNISUZUX0NPTFVNTlN9CiAgICBpZiBtaXNzaW5nIDo9IHNvcnRlZChyZXF1aXJlZF9tYXRyaXggLSBzZXQobWF0cml4X3Jvd3MuY29sdW1ucykpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJNYXRyaXggaXMgbWlzc2luZyBjb2x1bW5zOiB7bWlzc2luZ30iKQogICAgaWYgbWlzc2luZyA6PSBzb3J0ZWQocmVxdWlyZWRfc2hpZnRzIC0gc2V0KHNoaWZ0X3Jvd3MuY29sdW1ucykpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJTaGlmdCB0YWJsZSBpcyBtaXNzaW5nIGNvbHVtbnM6IHttaXNzaW5nfSIpCiAgICBtYXRyaXggPSBtYXRyaXhfcm93cy5jb3B5KCkKICAgIGlmICJzdGF0dXMiIGluIG1hdHJpeDoKICAgICAgICBtYXRyaXggPSBtYXRyaXgubG9jW21hdHJpeFsic3RhdHVzIl0uZXEoIkNPTVBMRVRFIildLmNvcHkoKQogICAgaWYgbWF0cml4LmR1cGxpY2F0ZWQobGlzdChQQUlSX0NPTFVNTlMpKS5hbnkoKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJNYXRyaXggY29udGFpbnMgZHVwbGljYXRlIHNvdXJjZS10YXJnZXQgY2VsbHMiKQogICAgc2hpZnRzID0gc2hpZnRfcm93cy5jb3B5KCkKICAgIGlmIHNoaWZ0cy5kdXBsaWNhdGVkKGxpc3QoUEFJUl9DT0xVTU5TKSkuYW55KCk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiU2hpZnQgdGFibGUgY29udGFpbnMgZHVwbGljYXRlIHNvdXJjZS10YXJnZXQgcm93cyIpCiAgICBmb3IgY29sdW1uIGluIFNISUZUX0NPTFVNTlM6CiAgICAgICAgdmFsdWVzID0gc2V0KHBkLnRvX251bWVyaWMoc2hpZnRzW2NvbHVtbl0sIGVycm9ycz0icmFpc2UiKS51bmlxdWUoKSkKICAgICAgICBpZiBub3QgdmFsdWVzLmlzc3Vic2V0KHswLCAxfSk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJ7Y29sdW1ufSBtdXN0IGJlIGJpbmFyeSIpCgogICAgZGlhZ29uYWwgPSBtYXRyaXgubG9jWwogICAgICAgIG1hdHJpeFsic291cmNlX2RhdGFzZXQiXS5lcShtYXRyaXhbInRhcmdldF9kYXRhc2V0Il0pLAogICAgICAgIFsic291cmNlX2RhdGFzZXQiLCAibWFjcm9fYXVyb2MiXSwKICAgIF0ucmVuYW1lKGNvbHVtbnM9eyJtYWNyb19hdXJvYyI6ICJpbl9kaXN0cmlidXRpb25fYXVyb2MifSkKICAgIGlmIGRpYWdvbmFsWyJzb3VyY2VfZGF0YXNldCJdLmR1cGxpY2F0ZWQoKS5hbnkoKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJNYXRyaXggY29udGFpbnMgZHVwbGljYXRlIGRpYWdvbmFsIGNlbGxzIikKICAgIG9mZiA9IG1hdHJpeC5sb2NbCiAgICAgICAgbWF0cml4WyJzb3VyY2VfZGF0YXNldCJdLm5lKG1hdHJpeFsidGFyZ2V0X2RhdGFzZXQiXSkKICAgIF0uY29weSgpCiAgICBjb21wb25lbnRzID0gb2ZmLm1lcmdlKGRpYWdvbmFsLCBvbj0ic291cmNlX2RhdGFzZXQiLCBob3c9ImxlZnQiLCB2YWxpZGF0ZT0ibWFueV90b19vbmUiKQogICAgY29tcG9uZW50cyA9IGNvbXBvbmVudHMubWVyZ2UoCiAgICAgICAgc2hpZnRzW2xpc3QoUEFJUl9DT0xVTU5TKSArIGxpc3QoU0hJRlRfQ09MVU1OUyldLAogICAgICAgIG9uPWxpc3QoUEFJUl9DT0xVTU5TKSwKICAgICAgICBob3c9ImxlZnQiLAogICAgICAgIHZhbGlkYXRlPSJvbmVfdG9fb25lIiwKICAgICkKICAgIGlmIGNvbXBvbmVudHMuZW1wdHk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiTm8gY29tcGxldGVkIG9mZi1kaWFnb25hbCBjZWxscyBhcmUgYXZhaWxhYmxlIikKICAgIGlmIGNvbXBvbmVudHNbWyJpbl9kaXN0cmlidXRpb25fYXVyb2MiLCAqU0hJRlRfQ09MVU1OU11dLmlzbmEoKS5hbnkoKS5hbnkoKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJFdmVyeSBvZmYtZGlhZ29uYWwgY2VsbCBuZWVkcyBhIGRpYWdvbmFsIGFuZCBzaGlmdCB2ZWN0b3IiKQoKICAgIGNvbXBvbmVudHNbImRlbHRhX2lqIl0gPSAoCiAgICAgICAgY29tcG9uZW50c1siaW5fZGlzdHJpYnV0aW9uX2F1cm9jIl0gLSBjb21wb25lbnRzWyJtYWNyb19hdXJvYyJdCiAgICApCiAgICBjb21wb25lbnRzWyJ3X2lqIl0gPSAoCiAgICAgICAgbGFtYmRhX3BzICogY29tcG9uZW50c1siUFMiXQogICAgICAgICsgbGFtYmRhX2RzICogY29tcG9uZW50c1siRFMiXQogICAgICAgICsgbGFtYmRhX2xzICogY29tcG9uZW50c1siTFMiXQogICAgKQogICAgY29tcG9uZW50c1sid2VpZ2h0ZWRfZ2FwX3Rlcm0iXSA9IGNvbXBvbmVudHNbImRlbHRhX2lqIl0gLyAoCiAgICAgICAgY29tcG9uZW50c1sid19paiJdICsgMS4wCiAgICApCiAgICBjcm9zc19kYXRhc2V0X21lYW4gPSBmbG9hdChjb21wb25lbnRzWyJtYWNyb19hdXJvYyJdLm1lYW4oKSkKICAgIHBlbmFsdHkgPSBmbG9hdChjb21wb25lbnRzWyJ3ZWlnaHRlZF9nYXBfdGVybSJdLm1lYW4oKSkKICAgIHNjb3JlID0gZmxvYXQobnAuY2xpcChjcm9zc19kYXRhc2V0X21lYW4gKiAoMS4wIC0gcGVuYWx0eSksIDAuMCwgMS4wKSkKICAgIGNvbXBvbmVudHNbImNyb3NzX2RhdGFzZXRfbWVhbiJdID0gY3Jvc3NfZGF0YXNldF9tZWFuCiAgICBjb21wb25lbnRzWyJwZW5hbHR5X21lYW4iXSA9IHBlbmFsdHkKICAgIGNvbXBvbmVudHNbImNvbXBvc2l0ZV9zY29yZSJdID0gc2NvcmUKICAgIHJldHVybiBzY29yZSwgY29tcG9uZW50cwoKCmRlZiBidWlsZF9wYXJzZXIoKSAtPiBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcjoKICAgIHBhcnNlciA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKGRlc2NyaXB0aW9uPV9fZG9jX18pCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW1hdHJpeCIsIHR5cGU9UGF0aCwgcmVxdWlyZWQ9VHJ1ZSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tc2hpZnQtdGFibGUiLCB0eXBlPVBhdGgsIHJlcXVpcmVkPVRydWUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW91dHB1dCIsIHR5cGU9UGF0aCwgcmVxdWlyZWQ9VHJ1ZSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tbGFtYmRhLXBzIiwgdHlwZT1mbG9hdCwgZGVmYXVsdD0xLjApCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWxhbWJkYS1kcyIsIHR5cGU9ZmxvYXQsIGRlZmF1bHQ9Mi4wKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1sYW1iZGEtbHMiLCB0eXBlPWZsb2F0LCBkZWZhdWx0PTMuMCkKICAgIHJldHVybiBwYXJzZXIKCgpkZWYgbWFpbihhcmd2OiBTZXF1ZW5jZVtzdHJdIHwgTm9uZSA9IE5vbmUpIC0+IGludDoKICAgIGFyZ3MgPSBidWlsZF9wYXJzZXIoKS5wYXJzZV9hcmdzKGFyZ3YpCiAgICBzY29yZSwgY29tcG9uZW50cyA9IGNvbXB1dGVfY29tcG9zaXRlX3Njb3JlKAogICAgICAgIHBkLnJlYWRfY3N2KGFyZ3MubWF0cml4KSwKICAgICAgICBwZC5yZWFkX2NzdihhcmdzLnNoaWZ0X3RhYmxlKSwKICAgICAgICBsYW1iZGFfcHM9YXJncy5sYW1iZGFfcHMsCiAgICAgICAgbGFtYmRhX2RzPWFyZ3MubGFtYmRhX2RzLAogICAgICAgIGxhbWJkYV9scz1hcmdzLmxhbWJkYV9scywKICAgICkKICAgIGFyZ3Mub3V0cHV0LnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBjb21wb25lbnRzLnRvX2NzdihhcmdzLm91dHB1dCwgaW5kZXg9RmFsc2UpCiAgICBwcmludCh7InNjb3JlIjogc2NvcmUsICJyb3dzIjogbGVuKGNvbXBvbmVudHMpLCAib3V0cHV0Ijogc3RyKGFyZ3Mub3V0cHV0KX0pCiAgICByZXR1cm4gMAoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICByYWlzZSBTeXN0ZW1FeGl0KG1haW4oKSkKCg==', 'tests/test_resnet1d.py': 'aW1wb3J0IHB5dGVzdAppbXBvcnQgdG9yY2gKCmZyb20gc3JjLm1vZGVscy5yZXNuZXQxZCBpbXBvcnQgUmVzTmV0MUQKCgpkZWYgdGVzdF9yZXNuZXQxZF9yZXR1cm5zX2ZpdmVfcmVjb3JkaW5nX2xvZ2l0cygpOgogICAgbW9kZWwgPSBSZXNOZXQxRChiYXNlX2NoYW5uZWxzPTQsIGJsb2Nrc19wZXJfc3RhZ2U9KDEsIDEsIDEsIDEpKQogICAgcmVzdWx0ID0gbW9kZWwodG9yY2gucmFuZG4oMiwgMTIsIDUwMDApKQogICAgYXNzZXJ0IHJlc3VsdC5zaGFwZSA9PSAoMiwgNSkKICAgIGFzc2VydCBhbGwocGFyYW1ldGVyLnJlcXVpcmVzX2dyYWQgZm9yIHBhcmFtZXRlciBpbiBtb2RlbC5wYXJhbWV0ZXJzKCkpCgoKZGVmIHRlc3RfcmVzbmV0MWRfcmVqZWN0c193cm9uZ19sZWFkX2NvdW50KCk6CiAgICBtb2RlbCA9IFJlc05ldDFEKGJhc2VfY2hhbm5lbHM9NCwgYmxvY2tzX3Blcl9zdGFnZT0oMSwgMSwgMSwgMSkpCiAgICB3aXRoIHB5dGVzdC5yYWlzZXMoVmFsdWVFcnJvciwgbWF0Y2g9ImJhdGNoLCAxMiwgc2FtcGxlcyIpOgogICAgICAgIG1vZGVsKHRvcmNoLnJhbmRuKDEsIDgsIDUwMDApKQoK', 'tests/test_transformer1d.py': 'aW1wb3J0IHB5dGVzdAppbXBvcnQgdG9yY2gKCmZyb20gc3JjLm1vZGVscy50cmFuc2Zvcm1lcjFkIGltcG9ydCBFQ0dUcmFuc2Zvcm1lcjFECgoKZGVmIHRlc3RfdHJhbnNmb3JtZXIxZF9yZXR1cm5zX2ZpdmVfcmVjb3JkaW5nX2xvZ2l0cygpOgogICAgbW9kZWwgPSBFQ0dUcmFuc2Zvcm1lcjFEKAogICAgICAgIHBhdGNoX3NpemU9MTAwLAogICAgICAgIGVtYmVkX2RpbT0xNiwKICAgICAgICBudW1faGVhZHM9MiwKICAgICAgICBudW1fbGF5ZXJzPTEsCiAgICAgICAgZmVlZGZvcndhcmRfZGltPTMyLAogICAgICAgIGRyb3BvdXQ9MC4wLAogICAgKQogICAgcmVzdWx0ID0gbW9kZWwodG9yY2gucmFuZG4oMiwgMTIsIDUwMDApKQogICAgYXNzZXJ0IHJlc3VsdC5zaGFwZSA9PSAoMiwgNSkKICAgIGFzc2VydCBhbGwocGFyYW1ldGVyLnJlcXVpcmVzX2dyYWQgZm9yIHBhcmFtZXRlciBpbiBtb2RlbC5wYXJhbWV0ZXJzKCkpCgoKZGVmIHRlc3RfdHJhbnNmb3JtZXIxZF9yZWplY3RzX3dyb25nX3dpbmRvd19sZW5ndGgoKToKICAgIG1vZGVsID0gRUNHVHJhbnNmb3JtZXIxRCgKICAgICAgICBlbWJlZF9kaW09MTYsIG51bV9oZWFkcz0yLCBudW1fbGF5ZXJzPTEsIGZlZWRmb3J3YXJkX2RpbT0zMgogICAgKQogICAgd2l0aCBweXRlc3QucmFpc2VzKFZhbHVlRXJyb3IsIG1hdGNoPSJFeHBlY3RlZCA1MDAwIHNhbXBsZXMiKToKICAgICAgICBtb2RlbCh0b3JjaC5yYW5kbigxLCAxMiwgMjUwMCkpCgo=', 'tests/test_baseline_pipeline.py': 'aW1wb3J0IHRvcmNoCmZyb20gdG9yY2ggaW1wb3J0IG5uCgpmcm9tIHNyYy50cmFpbmluZy5iYXNlbGluZV9waXBlbGluZSBpbXBvcnQgUmVjb3JkaW5nV2luZG93QWRhcHRlciwgYnVpbGRfYmFzZWxpbmVfbW9kZWwKCgpjbGFzcyBNZWFuUmVjb3JkaW5nTW9kZWwobm4uTW9kdWxlKToKICAgIGRlZiBmb3J3YXJkKHNlbGYsIHNpZ25hbCk6CiAgICAgICAgcmV0dXJuIHNpZ25hbC5tZWFuKGRpbT0oMSwgMiksIGtlZXBkaW09RmFsc2UpLnVuc3F1ZWV6ZSgxKQoKCmRlZiB0ZXN0X3JlY29yZGluZ19hZGFwdGVyX3JlYXNzZW1ibGVzX3dpbmRvd3NfYW5kX21hc2tzX3BhZGRpbmcoKToKICAgIGFkYXB0ZXIgPSBSZWNvcmRpbmdXaW5kb3dBZGFwdGVyKE1lYW5SZWNvcmRpbmdNb2RlbCgpKQogICAgc291cmNlID0gdG9yY2gub25lcygxLCAyLCAxMiwgMjUwMCkKICAgIHJlc3VsdCA9IGFkYXB0ZXIoc291cmNlLCB0b3JjaC50ZW5zb3IoW1tUcnVlLCBGYWxzZV1dKSkKICAgIGFzc2VydCB0b3JjaC5hbGxjbG9zZShyZXN1bHQsIHRvcmNoLnRlbnNvcihbWzAuNV1dKSkKCgpkZWYgdGVzdF9hbGxfYmFzZWxpbmVzX3JldHVybl9maXZlX2xvZ2l0cygpOgogICAgY29uZmlncyA9IHsKICAgICAgICAiaW5jZXB0aW9uX3RpbWUiOiB7ImluY2VwdGlvbl9jaGFubmVscyI6IDQsICJpbmNlcHRpb25fZGVwdGgiOiAzfSwKICAgICAgICAicmVzbmV0MWQiOiB7InJlc25ldF9iYXNlX2NoYW5uZWxzIjogNCwgInJlc25ldF9ibG9ja3MiOiBbMSwgMSwgMSwgMV19LAogICAgICAgICJ0cmFuc2Zvcm1lciI6IHsKICAgICAgICAgICAgInRyYW5zZm9ybWVyX3BhdGNoX3NpemUiOiAxMDAsCiAgICAgICAgICAgICJ0cmFuc2Zvcm1lcl9lbWJlZF9kaW0iOiAxNiwKICAgICAgICAgICAgInRyYW5zZm9ybWVyX2hlYWRzIjogMiwKICAgICAgICAgICAgInRyYW5zZm9ybWVyX2xheWVycyI6IDEsCiAgICAgICAgICAgICJ0cmFuc2Zvcm1lcl9mZWVkZm9yd2FyZF9kaW0iOiAzMiwKICAgICAgICB9LAogICAgfQogICAgc291cmNlID0gdG9yY2gucmFuZG4oMiwgMiwgMTIsIDI1MDApCiAgICBtYXNrID0gdG9yY2gub25lcygyLCAyLCBkdHlwZT10b3JjaC5ib29sKQogICAgZm9yIGFyY2hpdGVjdHVyZSwgY29uZmlnIGluIGNvbmZpZ3MuaXRlbXMoKToKICAgICAgICBtb2RlbCwgcG9saWN5ID0gYnVpbGRfYmFzZWxpbmVfbW9kZWwoYXJjaGl0ZWN0dXJlLCBtb2RlbF9jb25maWc9Y29uZmlnKQogICAgICAgIGFzc2VydCBtb2RlbChzb3VyY2UsIG1hc2spLnNoYXBlID09ICgyLCA1KQogICAgICAgIGFzc2VydCBwb2xpY3lbImZyb3plbl9wYXJhbWV0ZXJfY291bnQiXSA9PSAwCgo=', 'tests/test_composite_score.py': 'aW1wb3J0IHBhbmRhcyBhcyBwZAppbXBvcnQgcHl0ZXN0Cgpmcm9tIHNyYy5ldmFsdWF0aW9uLmNvbXBvc2l0ZV9zY29yZSBpbXBvcnQgY29tcHV0ZV9jb21wb3NpdGVfc2NvcmUKCgpkZWYgdGVzdF9wcm9wb3NhbF9jb21wb3NpdGVfc2NvcmVfbWF0Y2hlc19oYW5kX2NhbGN1bGF0aW9uKCk6CiAgICBtYXRyaXggPSBwZC5EYXRhRnJhbWUoCiAgICAgICAgWwogICAgICAgICAgICB7InNvdXJjZV9kYXRhc2V0IjogImEiLCAidGFyZ2V0X2RhdGFzZXQiOiAiYSIsICJtYWNyb19hdXJvYyI6IDAuOX0sCiAgICAgICAgICAgIHsic291cmNlX2RhdGFzZXQiOiAiYSIsICJ0YXJnZXRfZGF0YXNldCI6ICJiIiwgIm1hY3JvX2F1cm9jIjogMC43fSwKICAgICAgICAgICAgeyJzb3VyY2VfZGF0YXNldCI6ICJiIiwgInRhcmdldF9kYXRhc2V0IjogImEiLCAibWFjcm9fYXVyb2MiOiAwLjZ9LAogICAgICAgICAgICB7InNvdXJjZV9kYXRhc2V0IjogImIiLCAidGFyZ2V0X2RhdGFzZXQiOiAiYiIsICJtYWNyb19hdXJvYyI6IDAuOH0sCiAgICAgICAgXQogICAgKQogICAgc2hpZnRzID0gcGQuRGF0YUZyYW1lKAogICAgICAgIFsKICAgICAgICAgICAgeyJzb3VyY2VfZGF0YXNldCI6ICJhIiwgInRhcmdldF9kYXRhc2V0IjogImIiLCAiUFMiOiAxLCAiRFMiOiAwLCAiTFMiOiAwfSwKICAgICAgICAgICAgeyJzb3VyY2VfZGF0YXNldCI6ICJiIiwgInRhcmdldF9kYXRhc2V0IjogImEiLCAiUFMiOiAwLCAiRFMiOiAxLCAiTFMiOiAxfSwKICAgICAgICBdCiAgICApCiAgICBzY29yZSwgY29tcG9uZW50cyA9IGNvbXB1dGVfY29tcG9zaXRlX3Njb3JlKG1hdHJpeCwgc2hpZnRzKQogICAgIyBDRD0uNjU7IHBlbmFsdHk9bWVhbiguMi8oMSsxKSwgLjIvKDUrMSkpPS4wNjY2NjYuLi4KICAgIGFzc2VydCBzY29yZSA9PSBweXRlc3QuYXBwcm94KDAuNjUgKiAoMS4wIC0gKDAuMSArIDEgLyAzMCkgLyAyKSkKICAgIGFzc2VydCBjb21wb25lbnRzWyJkZWx0YV9paiJdLnRvbGlzdCgpID09IHB5dGVzdC5hcHByb3goWzAuMiwgMC4yXSkKICAgIGFzc2VydCBjb21wb25lbnRzWyJ3X2lqIl0udG9saXN0KCkgPT0gcHl0ZXN0LmFwcHJveChbMS4wLCA1LjBdKQoKCmRlZiB0ZXN0X2NvbXBvc2l0ZV9zY29yZV9yZXF1aXJlc19jb21wbGV0ZV9zaGlmdF92ZWN0b3JzKCk6CiAgICBtYXRyaXggPSBwZC5EYXRhRnJhbWUoCiAgICAgICAgWwogICAgICAgICAgICB7InNvdXJjZV9kYXRhc2V0IjogImEiLCAidGFyZ2V0X2RhdGFzZXQiOiAiYSIsICJtYWNyb19hdXJvYyI6IDAuOX0sCiAgICAgICAgICAgIHsic291cmNlX2RhdGFzZXQiOiAiYSIsICJ0YXJnZXRfZGF0YXNldCI6ICJiIiwgIm1hY3JvX2F1cm9jIjogMC43fSwKICAgICAgICBdCiAgICApCiAgICBzaGlmdHMgPSBwZC5EYXRhRnJhbWUoY29sdW1ucz1bInNvdXJjZV9kYXRhc2V0IiwgInRhcmdldF9kYXRhc2V0IiwgIlBTIiwgIkRTIiwgIkxTIl0pCiAgICB3aXRoIHB5dGVzdC5yYWlzZXMoVmFsdWVFcnJvciwgbWF0Y2g9InNoaWZ0IHZlY3RvciIpOgogICAgICAgIGNvbXB1dGVfY29tcG9zaXRlX3Njb3JlKG1hdHJpeCwgc2hpZnRzKQoK'}
for relative_path, encoded in EMBEDDED_FILES.items():
    destination = WORKSPACE / relative_path
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_bytes(base64.b64decode(encoded))

subprocess.check_call([
    sys.executable, "-m", "pytest", "-q",
    "tests/test_resnet1d.py",
    "tests/test_transformer1d.py",
    "tests/test_baseline_pipeline.py",
    "tests/test_composite_score.py",
], cwd=WORKSPACE)
print({"embedded_files": len(EMBEDDED_FILES), "architecture_unit_tests": "PASS"})


In [ ]:
#@title 4. Resolve manifests and prepare waveform roots
import shutil
import tarfile

DATASET_FOLDERS = {
    "ptbxl": DATASETS_ROOT / "01_PTB_XL__21K_RECORDINGS",
    "cpsc2018": DATASETS_ROOT / "02_CPSC_2018__FULL_DATA_STILL_NEEDED",
    "georgia": DATASETS_ROOT / "03_GEORGIA_12_LEAD__10K_READY",
    "mimic_iv": DATASETS_ROOT / "04_MIMIC_IV_ECG__50K_READY",
    "code_ii": DATASETS_ROOT / "05_CODE_15_PERCENT__60K_POOL_NEEDS_FINAL_50K",
}

drive_manifest_root = CODE_ROOT / "03_ECG_FM_TRAINING_AND_EVALUATION" / "manifests"
MANIFEST_ROOT = drive_manifest_root if drive_manifest_root.is_dir() else WORKSPACE / "data" / "week2"
LOCAL_DATA_ROOT = Path("/content/ecg-benchmark-data")
LOCAL_DATA_ROOT.mkdir(parents=True, exist_ok=True)

def copy_npy_dataset(name):
    source = DATASET_FOLDERS[name]
    if not source.is_dir():
        return None
    if not STAGE_NPY_SIGNALS_TO_LOCAL_DISK:
        return source
    destination = LOCAL_DATA_ROOT / name
    target_signals = destination / "signals"
    if not target_signals.is_dir():
        source_signals = source / "signals"
        if not source_signals.is_dir():
            return source
        destination.mkdir(parents=True, exist_ok=True)
        shutil.copytree(source_signals, target_signals, dirs_exist_ok=True)
    return destination

def prepare_georgia():
    source = DATASET_FOLDERS["georgia"]
    destination = LOCAL_DATA_ROOT / "georgia"
    if (destination / "signals").is_dir():
        return destination
    if not RUN_DATA_PREPARATION:
        return destination
    archive = Path("/content/georgia_signals.tar.gz")
    with archive.open("wb") as output:
        for part_number in range(7):
            whole = source / f"georgia_signals.tar.gz.part-{part_number:02d}"
            parts = [whole] if whole.exists() else sorted(
                source.glob(f"georgia_signals.tar.gz.part-{part_number:02d}.sub-*")
            )
            if not parts:
                raise FileNotFoundError(f"Missing Georgia archive part {part_number:02d}")
            for part in parts:
                with part.open("rb") as handle:
                    shutil.copyfileobj(handle, output, length=8 * 1024 * 1024)
    destination.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive, "r:gz") as handle:
        handle.extractall(destination, filter="data")
    return destination

def prepare_mimic():
    source = DATASET_FOLDERS["mimic_iv"] / "mimic_50k_waveforms"
    destination = LOCAL_DATA_ROOT / "mimic_iv"
    candidates = [destination / "mimic_50k_waveforms", destination]
    ready = next((candidate for candidate in candidates if (candidate / "files").is_dir()), None)
    if ready is not None:
        return ready
    if not RUN_DATA_PREPARATION:
        return destination / "mimic_50k_waveforms"
    destination.mkdir(parents=True, exist_ok=True)
    shards = sorted(source.glob("mimic_50k_waveforms_*-of-050.tar.gz"))
    if len(shards) != 50:
        raise FileNotFoundError(f"Expected 50 MIMIC waveform shards, found {len(shards)}")
    for archive in shards:
        with tarfile.open(archive, "r:gz") as handle:
            handle.extractall(destination, filter="data")
    overlay = source / "mimic_50k_v3_replacement_overlay"
    if overlay.is_dir():
        for archive in sorted(overlay.glob("*.tar.gz")):
            with tarfile.open(archive, "r:gz") as handle:
                handle.extractall(destination, filter="data")
    ready = next((candidate for candidate in candidates if (candidate / "files").is_dir()), None)
    if ready is None:
        raise FileNotFoundError("MIMIC shards extracted, but no files/ directory was found")
    return ready

SIGNAL_ROOTS = {
    "ptbxl": copy_npy_dataset("ptbxl"),
    "cpsc2018": copy_npy_dataset("cpsc2018"),
    "georgia": prepare_georgia(),
    "mimic_iv": prepare_mimic(),
}

print({
    "manifest_root": str(MANIFEST_ROOT),
    "signal_roots": {name: str(path) for name, path in SIGNAL_ROOTS.items() if path is not None},
})


In [ ]:
#@title 5. Full manifest-to-waveform readiness audit
import pandas as pd

def waveform_exists(root, row):
    relative = Path(str(row["signal_path"]))
    path = root / relative
    storage = str(row["storage"])
    if storage == "npy":
        return path.is_file()
    if storage == "wfdb":
        base = path.with_suffix("") if path.suffix in {".hea", ".dat"} else path
        return base.with_suffix(".hea").is_file() and base.with_suffix(".dat").is_file()
    return False

readiness_rows = []
for dataset in REQUESTED_DATASETS:
    manifest_path = MANIFEST_ROOT / f"{dataset}_week2.csv"
    root = SIGNAL_ROOTS.get(dataset)
    if not manifest_path.is_file():
        readiness_rows.append({"dataset": dataset, "ready": False, "records": 0, "missing_waveforms": None, "reason": "missing frozen manifest"})
        continue
    if root is None or not root.exists():
        readiness_rows.append({"dataset": dataset, "ready": False, "records": 0, "missing_waveforms": None, "reason": "missing waveform root"})
        continue
    manifest = pd.read_csv(manifest_path, low_memory=False)
    required = {"signal_path", "storage", "split"}
    missing_columns = sorted(required - set(manifest.columns))
    if missing_columns:
        readiness_rows.append({"dataset": dataset, "ready": False, "records": len(manifest), "missing_waveforms": None, "reason": f"manifest missing columns: {missing_columns}"})
        continue
    missing_count = sum(not waveform_exists(root, row) for _, row in manifest.iterrows())
    readiness_rows.append({
        "dataset": dataset,
        "ready": missing_count == 0 and len(manifest) > 0,
        "records": len(manifest),
        "missing_waveforms": missing_count,
        "reason": "ready" if missing_count == 0 else "manifest references missing waveform files",
    })

readiness = pd.DataFrame(readiness_rows)
display(readiness)
READY_DATASETS = readiness.loc[readiness["ready"], "dataset"].tolist()
BLOCKED_DATASETS = readiness.loc[~readiness["ready"], "dataset"].tolist()

if STRICTLY_REQUIRE_ALL_FIVE_DATASETS and BLOCKED_DATASETS:
    raise RuntimeError(f"All-five-dataset run blocked by: {BLOCKED_DATASETS}")
if not READY_DATASETS:
    raise RuntimeError("No dataset passed the readiness audit")

print({"ready_datasets": READY_DATASETS, "blocked_datasets": BLOCKED_DATASETS})


In [ ]:
#@title 6. Download and verify the ECG-FM checkpoint
from huggingface_hub import hf_hub_download
import torch

ECG_FM_CHECKPOINT = Path(hf_hub_download(
    repo_id="wanglab/ecg-fm",
    filename="mimic_iv_ecg_physionet_pretrained.pt",
    local_dir="/content/checkpoints/ecg-fm",
))

if ECG_FM_CHECKPOINT.stat().st_size < 100_000_000:
    raise RuntimeError("Downloaded ECG-FM checkpoint is unexpectedly small")

print({
    "checkpoint": str(ECG_FM_CHECKPOINT),
    "size_mb": round(ECG_FM_CHECKPOINT.stat().st_size / 1e6, 1),
    "cuda": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
})


In [ ]:
#@title 7. Resumable command runner and output layout
import datetime as dt
import json

MASTER_RUN_ROOT = RESULTS_ROOT / "MASTER_COLAB_RUNS" / MODE
FIVE_CLASS_RUNS = MASTER_RUN_ROOT / "01_FIVE_CLASS_CHECKPOINTS"
FIVE_CLASS_MATRICES = MASTER_RUN_ROOT / "02_FIVE_CLASS_MATRICES"
BINARY_RUNS = MASTER_RUN_ROOT / "03_TWO_CLASS_CHECKPOINTS"
BINARY_MATRIX = MASTER_RUN_ROOT / "04_TWO_CLASS_MATRICES"
STATUS_PATH = MASTER_RUN_ROOT / "master_run_status.json"
MASTER_RUN_ROOT.mkdir(parents=True, exist_ok=True)

run_status = []

def run_command(label, command, sentinel=None):
    if RESUME_COMPLETED_RUNS and sentinel is not None and Path(sentinel).exists():
        row = {"label": label, "status": "SKIPPED_ALREADY_COMPLETE", "sentinel": str(sentinel)}
        run_status.append(row)
        print(row)
        return
    print("\nRUNNING:", label)
    print(" ".join(map(str, command)))
    started = dt.datetime.now(dt.timezone.utc)
    try:
        subprocess.check_call(list(map(str, command)), cwd=WORKSPACE)
        state = "COMPLETE"
        detail = ""
    except Exception as exc:
        state = "FAILED"
        detail = repr(exc)
        raise
    finally:
        run_status.append({
            "label": label,
            "status": state,
            "detail": detail,
            "started_utc": started.isoformat(),
            "finished_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
        })
        STATUS_PATH.write_text(json.dumps(run_status, indent=2) + "\n")

def signal_root_arguments(datasets):
    values = []
    for dataset in datasets:
        values.extend(["--signal-root", f"{dataset}={SIGNAL_ROOTS[dataset]}"])
    return values

print({"master_run_root": str(MASTER_RUN_ROOT), "status_file": str(STATUS_PATH)})


In [ ]:
#@title 8. Run signal/data sanity checks
if RUN_SANITY_CHECKS:
    for dataset in READY_DATASETS:
        command = [
            sys.executable, "-m", "src.training.ecg_fm_pipeline",
            "--dataset", dataset,
            "--manifest", MANIFEST_ROOT / f"{dataset}_week2.csv",
            "--signal-root", SIGNAL_ROOTS[dataset],
            "--output-dir", MASTER_RUN_ROOT / "00_DATA_CHECKS" / dataset,
            "--data-only", "--max-records-per-split", "5",
        ]
        run_command(f"sanity::{dataset}", command)
else:
    print("Sanity checks disabled")


In [ ]:
#@title 9. Train all four five-class architectures on every ready source
if RUN_ALL_5_CLASS_TRAINING:
    for architecture in FIVE_CLASS_ARCHITECTURES:
        for dataset in READY_DATASETS:
            output_dir = FIVE_CLASS_RUNS / architecture / dataset
            if architecture == "ecg_fm":
                command = [
                    sys.executable, "-m", "src.training.ecg_fm_pipeline",
                    "--dataset", dataset,
                    "--manifest", MANIFEST_ROOT / f"{dataset}_week2.csv",
                    "--signal-root", SIGNAL_ROOTS[dataset],
                    "--pretrained-checkpoint", ECG_FM_CHECKPOINT,
                    "--output-dir", output_dir,
                    "--seed", str(SEED),
                ]
                if MODE == "smoke":
                    command += [
                        "--smoke-test", "--epochs", "1", "--patience", "1",
                        "--max-records-per-split", "128", "--batch-size", "2",
                        "--gradient-accumulation-steps", "1",
                    ]
                else:
                    command += [
                        "--epochs", "50", "--patience", "10", "--batch-size", "4",
                        "--gradient-accumulation-steps", "8", "--learning-rate", "1e-6",
                    ]
            else:
                command = [
                    sys.executable, "-m", "src.training.baseline_pipeline",
                    "--architecture", architecture,
                    "--dataset", dataset,
                    "--manifest", MANIFEST_ROOT / f"{dataset}_week2.csv",
                    "--signal-root", SIGNAL_ROOTS[dataset],
                    "--output-dir", output_dir,
                    "--seed", str(SEED),
                ]
                if MODE == "smoke":
                    command += [
                        "--smoke-test", "--epochs", "1", "--patience", "1",
                        "--max-records-per-split", "128", "--batch-size", "8",
                        "--gradient-accumulation-steps", "1", "--no-mixed-precision",
                    ]
                    if architecture == "inception_time":
                        command += ["--inception-channels", "4", "--inception-depth", "3"]
                    elif architecture == "resnet1d":
                        command += ["--resnet-base-channels", "4", "--resnet-blocks", "1", "1", "1", "1"]
                    elif architecture == "transformer":
                        command += [
                            "--transformer-patch-size", "100", "--transformer-embed-dim", "16",
                            "--transformer-heads", "2", "--transformer-layers", "1",
                            "--transformer-feedforward-dim", "32",
                        ]
                else:
                    command += ["--epochs", "50", "--patience", "10"]
                    if architecture == "transformer":
                        command += ["--batch-size", "16", "--learning-rate", "3e-4"]
                    else:
                        command += ["--batch-size", "32", "--learning-rate", "1e-3"]
            run_command(
                f"five_class_train::{architecture}::{dataset}",
                command,
                sentinel=output_dir / "best_checkpoint.pt",
            )
else:
    print("All five-class training disabled")


In [ ]:
#@title 10. Run all four source-by-target matrices
if RUN_ALL_5_CLASS_MATRICES:
    for architecture in FIVE_CLASS_ARCHITECTURES:
        output_dir = FIVE_CLASS_MATRICES / architecture
        matrix_csv = output_dir / f"{architecture}_five_label_matrix_long.csv"
        if architecture == "ecg_fm":
            command = [
                sys.executable, "-m", "src.evaluation.ecg_fm_matrix",
                "--source-runs-root", FIVE_CLASS_RUNS / architecture,
                "--manifest-root", MANIFEST_ROOT,
                "--pretrained-checkpoint", ECG_FM_CHECKPOINT,
                "--output-dir", output_dir,
                "--datasets", *READY_DATASETS,
                "--seed", str(SEED),
                *signal_root_arguments(READY_DATASETS),
            ]
        else:
            command = [
                sys.executable, "-m", "src.evaluation.baseline_matrix",
                "--architecture", architecture,
                "--source-runs-root", FIVE_CLASS_RUNS / architecture,
                "--manifest-root", MANIFEST_ROOT,
                "--output-dir", output_dir,
                "--datasets", *READY_DATASETS,
                "--seed", str(SEED),
                *signal_root_arguments(READY_DATASETS),
            ]
        if MODE == "smoke":
            command += ["--max-records-per-target", "128", "--batch-size", "2", "--no-mixed-precision"]
        run_command(f"five_class_matrix::{architecture}", command, sentinel=matrix_csv)
else:
    print("All five-class matrices disabled")


In [ ]:
#@title 10b. Proposal Score for every completed architecture matrix
if RUN_COMPOSITE_SCORE:
    if SHIFT_TABLE_OVERRIDE:
        SHIFT_TABLE = Path(SHIFT_TABLE_OVERRIDE)
    else:
        shift_candidates = list(PROJECT_ROOT.rglob("shift_vectors.csv")) + list(PROJECT_ROOT.rglob("*shift*vector*.csv"))
        SHIFT_TABLE = shift_candidates[0] if shift_candidates else None
    if SHIFT_TABLE is None or not SHIFT_TABLE.is_file():
        print("Composite Score blocked: no shift_vectors.csv was found. Set SHIFT_TABLE_OVERRIDE in cell 1.")
    else:
        for architecture in FIVE_CLASS_ARCHITECTURES:
            matrix_csv = FIVE_CLASS_MATRICES / architecture / f"{architecture}_five_label_matrix_long.csv"
            if not matrix_csv.is_file():
                print(f"Score skipped for {architecture}: missing {matrix_csv}")
                continue
            output_csv = FIVE_CLASS_MATRICES / architecture / "composite_score_components.csv"
            command = [
                sys.executable, "-m", "src.evaluation.composite_score",
                "--matrix", matrix_csv,
                "--shift-table", SHIFT_TABLE,
                "--output", output_csv,
            ]
            run_command(f"composite_score::{architecture}", command, sentinel=output_csv)
else:
    print("Composite Score disabled")


In [ ]:
#@title 11. Train both two-class ablation architectures
if RUN_2_CLASS_TRAINING:
    for architecture in BINARY_ARCHITECTURES:
        for dataset in READY_DATASETS:
            output_dir = BINARY_RUNS / architecture / dataset
            command = [
                sys.executable, "-m", "src.training.binary_ablation_pipeline",
                "--architecture", architecture,
                "--manifest", MANIFEST_ROOT / f"{dataset}_week2.csv",
                "--signal-root", SIGNAL_ROOTS[dataset],
                "--output-dir", output_dir,
                "--seed", str(SEED),
            ]
            if architecture == "ecg_fm":
                command += ["--pretrained-checkpoint", ECG_FM_CHECKPOINT]
            if MODE == "smoke":
                command += [
                    "--smoke-test", "--epochs", "1", "--patience", "1",
                    "--max-records-per-split", "128", "--batch-size", "2",
                    "--gradient-accumulation-steps", "1",
                ]
            else:
                command += ["--epochs", "50", "--patience", "10"]
                if architecture == "ecg_fm":
                    command += ["--batch-size", "4", "--gradient-accumulation-steps", "8", "--learning-rate", "1e-6"]
                else:
                    command += ["--batch-size", "32", "--gradient-accumulation-steps", "1", "--learning-rate", "1e-3"]
            run_command(
                f"two_class_train::{architecture}::{dataset}",
                command,
                sentinel=output_dir / "best_checkpoint.pt",
            )
else:
    print("Two-class training disabled")


In [ ]:
#@title 12. Run both two-class matrices and gap-disappearance analysis
if RUN_2_CLASS_MATRIX:
    matrix_csv = BINARY_MATRIX / "binary_matrix_long.csv"
    command = [
        sys.executable, "-m", "src.evaluation.binary_matrix",
        "--manifest-root", MANIFEST_ROOT,
        "--source-runs-root", BINARY_RUNS,
        "--pretrained-checkpoint", ECG_FM_CHECKPOINT,
        "--output-dir", BINARY_MATRIX,
        "--datasets", *READY_DATASETS,
        "--architectures", *BINARY_ARCHITECTURES,
        "--seed", str(SEED),
        *signal_root_arguments(READY_DATASETS),
    ]
    for architecture in BINARY_ARCHITECTURES:
        five_class_csv = FIVE_CLASS_MATRICES / architecture / f"{architecture}_five_label_matrix_long.csv"
        if five_class_csv.is_file():
            command += ["--five-label-matrix", f"{architecture}={five_class_csv}"]
    if MODE == "smoke":
        command += ["--max-records-per-target", "128", "--batch-size", "2"]
    run_command("two_class_matrix::ecg_fm_and_inception_time", command, sentinel=matrix_csv)
else:
    print("Two-class matrix disabled")


In [ ]:
#@title 13. Final experiment summary
summary_files = {
    "dataset_readiness": MASTER_RUN_ROOT / "dataset_readiness.csv",
    "binary_matrix": BINARY_MATRIX / "binary_matrix_long.csv",
    "gap_disappearance": BINARY_MATRIX / "two_class_gap_summary.csv",
    "run_status": STATUS_PATH,
}
for architecture in FIVE_CLASS_ARCHITECTURES:
    summary_files[f"five_class_matrix_{architecture}"] = (
        FIVE_CLASS_MATRICES / architecture / f"{architecture}_five_label_matrix_long.csv"
    )

readiness.to_csv(summary_files["dataset_readiness"], index=False)
summary = []
for name, path in summary_files.items():
    summary.append({"output": name, "exists": path.exists(), "path": str(path)})
display(pd.DataFrame(summary))

diagonal_rows = []
for architecture in FIVE_CLASS_ARCHITECTURES:
    path = summary_files[f"five_class_matrix_{architecture}"]
    if not path.is_file():
        continue
    frame = pd.read_csv(path)
    diagonal = frame.loc[frame["source_dataset"].eq(frame["target_dataset"])].copy()
    diagonal_rows.append(diagonal)
if diagonal_rows:
    diagonals = pd.concat(diagonal_rows, ignore_index=True)
    diagonals.to_csv(MASTER_RUN_ROOT / "all_architectures_in_distribution_diagonals.csv", index=False)
    display(diagonals[["architecture", "source_dataset", "macro_auroc", "status"]])

print("READY DATASETS:", READY_DATASETS)
print("BLOCKED DATASETS:", BLOCKED_DATASETS)
print("All outputs are saved under:", MASTER_RUN_ROOT)
print("Use MODE='full' only after the smoke run completes successfully.")
